In [ ]:
from typing import Literal, Tuple
from anndata import AnnData
from tqdm.auto import tqdm
from pycirclize import Circos
from bokeh.palettes import all_palettes

import re
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.colorbar as colorbar

In [ ]:
celltype_color_mapping = {
    'B cell': "#2278B5",
    'B Naive': "#2278B5",
    'B Intermediate/Memory': "#42B9EB",
    
    'CD4 T cell': "#F57F20",
    'CD4 Naive': "#F7901E",
    'CD4 Memory': "#F15A22",

    'CD8 T cell': "#2FA148",
    'CD8 Naive': "#2FA148", #"#73BF98", #"#74AC42",
    'CD8 Memory': "#17A57F",
    
    'Treg': "#7750A1",
    'MAIT': "#4C87C6",
    'NKT': "#ED2224",
    'NK CD56hi': "#F9BFCB",
    'CD56-bright NK': "#F9BFCB",
    'NK CD56low': "#8C471F",
    'CD56-dim NK': "#8C471F",
    'ILC': "#282A74",
    'pDC': "#ED2A91",
    'cDC': "#6ABD45",
    'CD14 Mono': "#7D277D",
    'CD16 Mono': "#E4C494",
    'HSPC': "#6A6A6A",
    "Granulocyte": "#FDD700",
    "other": "#FDD700",
    "Plasmablast": "#556B30",
    # "Platelet": "#FFFFFF",
    "Platelet": '#d9d9d9',
    
}

In [ ]:
def clean_cytokine_label(x):
    x = x.replace("alpha", r"$\alpha$")
    x = x.replace("beta", r"$\beta$")
    x = x.replace("gamma", r"$\gamma$")
    x = x.replace("omega", r"$\omega$")
    return x


def clean_celltype_label(x):
    if isinstance(x, list):
        return [clean_celltype_label(_x) for _x in x]
    text = x.get_text() if hasattr(x, 'get_text') else x

    if not text.startswith("NK"):
        text = text.split("(", 1)[0]
    else:
        text = text.replace("CD56-dim_NK", "CD56low")
        text = text.replace("CD56-bright_NK", "CD56high")
    
    return text


def _check_robustness_fractions(
    df_pivot, 
    threshold_qval=0.1, # adjusted p value
    threshold_valid=0.5, # fraction of results required to even consider this condition. I.e. if the test only ran for one set of thresholds, then it is not very robust.
    threshold_below_alpha=0.75, # fraction of results that need to be significant
):
    n_total = np.prod(df_pivot.shape)
    n_valid = n_total - df_pivot.isna().sum().sum()
    n_below_alpha = (df_pivot < threshold_qval).sum().sum() # number of results below pval threshold, i.e., number of significant results
    frac_valid_results = (n_valid / n_total)
    frac_pval_below_alpha = (n_below_alpha / n_valid) # fraction of significant results relative to valid results
    is_robust = (frac_pval_below_alpha > threshold_below_alpha) & (frac_valid_results > threshold_valid)
    return frac_valid_results, frac_pval_below_alpha, is_robust


def check_robustness(
    all_results,
    THRESHOLD_QVAL = 0.1,
    THRESHOLD_VALID = 0.5,
    THRESHOLD_BELOW_ALPHA = 0.9,
):
    
    all_thresholds_expression = all_results.threshold_expression.sort_values(ascending = False).unique()
    all_thresholds_lfc = sorted(all_results.threshold_lfc.unique())
    
    df = pd.DataFrame(index=all_thresholds_expression, columns=all_thresholds_lfc)
    df.index.rename("threshold_expression", inplace=True)
    df.columns.rename("threshold_lfc", inplace=True)

    df2 = df.copy()
    
    robust_results = []
    
    for contrast in tqdm(all_results.contrast.unique()):
        for celltype_combo in all_results.celltype_combo.unique():
            results_ct = all_results.loc[(all_results.celltype_combo == celltype_combo) & (all_results.contrast == contrast)]
            for cytokine in results_ct.cytokine.unique():
                results_ct_cy = results_ct.loc[results_ct.cytokine == cytokine]
                df_pivot = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="FDR q-val")
                df_pivot_nes = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="NES")
                row_order = df_pivot.index.sort_values(ascending=False)
                with warnings.catch_warnings():
                    warnings.simplefilter(action='ignore', category=FutureWarning)
                    df_combined = pd.concat([df, df_pivot])
                    df_combined2 = pd.concat([df2, df_pivot_nes])
                df_merged = df_combined.combine_first(df_pivot)
                df_merged2 = df_combined2.combine_first(df_pivot_nes)
                df_merged = df_merged.loc[~df_merged.index.duplicated()]
                df_merged2 = df_merged2.loc[~df_merged2.index.duplicated()]
                df_pivot = df_merged.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                df_pivot_nes = df_merged2.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                frac_valid_results, frac_pval_below_alpha, is_robust = _check_robustness_fractions(
                    df_pivot,
                    threshold_qval=THRESHOLD_QVAL,
                    threshold_valid=THRESHOLD_VALID,
                    threshold_below_alpha=THRESHOLD_BELOW_ALPHA
                )
                
                if is_robust:
                    # print(df_pivot_nes.max().max(), df_pivot_nes.min().min())
                    if (df_pivot_nes.max().max() > 0) and (df_pivot_nes.min().min() < 0):
                        sns.heatmap(df_pivot, cmap="Reds_r", center=0, vmin=0, vmax=0.1, square=True)
                        plt.show()
                        sns.heatmap(df_pivot_nes, cmap="RdBu_r", center=0, square=True)
                        plt.show()
                    robust_results.append(
                        (
                            celltype_combo, contrast, cytokine,
                            frac_valid_results, frac_pval_below_alpha, is_robust,
                            results_ct_cy.NES.min(), results_ct_cy.NES.max(),
                            THRESHOLD_QVAL,
                            THRESHOLD_BELOW_ALPHA,
                        )
                    )
        
    robust_results = pd.DataFrame(robust_results).rename(
        {0: "celltype_combo", 1: "contrast", 2: "cytokine", 3: "frac_valid", 4: "frac_significant", 5:"is_robust", 6:"NES_min", 7:"NES_max", 8:"qval_threshold", 9: "threshold_frac_below_alpha"},
        axis=1
    )
    return robust_results


def get_robust_significant_results(
    results,
    alphas,
    threshold_valid,
    threshold_below_alpha,
):

    results_robust = []
    for alpha in alphas:
        results_robust.append(check_robustness(results, THRESHOLD_QVAL=alpha, THRESHOLD_VALID=threshold_valid, THRESHOLD_BELOW_ALPHA=threshold_below_alpha))
    results_robust = pd.concat(results_robust)
    results_robust = results_robust.groupby(["contrast", "celltype_combo", "cytokine"])["qval_threshold"].min().to_frame().reset_index()
    
    # results_mean = results.fillna(0).groupby(["contrast", "celltype_combo", "cytokine"])["NES"].mean().to_frame().reset_index()
    results_mean = results.groupby(["contrast", "celltype_combo", "cytokine"])["NES"].mean().to_frame().reset_index()
    results_pivot = results_mean.pivot(
        index="cytokine",
        columns="celltype_combo",
        values="NES",
    )
    df_annot = results_pivot.copy()
    with warnings.catch_warnings():
        warnings.simplefilter(action='ignore', category=FutureWarning)
        df_annot[:] = ""
    for cytokine in df_annot.index:
        for celltype in df_annot.columns:
            qval = results_robust.loc[(results_robust.cytokine == cytokine) & (results_robust.celltype_combo == celltype)].qval_threshold
            if len(qval) != 0:
                qval = qval.values[0]
                if qval == 0.1:
                    df_annot.loc[cytokine, celltype] = "*"
                if qval == 0.05:
                    df_annot.loc[cytokine, celltype] = "**"
                if qval == 0.01:
                    df_annot.loc[cytokine, celltype] = "***"

    return results_pivot, df_annot, results_robust


def replace_greek(text):
    text = text.replace("beta", r"$\beta$")
    text = text.replace("alpha", r"$\alpha$")
    text = text.replace("gamma", r"$\gamma$")
    text = text.replace("lambda", r"$\lambda$")
    text = text.replace("omega", r"$\omega$")
    return text


def format_cytokine_names(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_cytokine_names(_x) for _x in x]
    text = x.get_text() if hasattr(x, 'get_text') else x
    return replace_greek(text)


def format_celltype_names(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_celltype_names(_x) for _x in x]
    text = x.get_text() if hasattr(x, 'get_text') else x
    # text = text.replace("CD4-positive, alpha-beta T cell", r"CD4, $\alpha$-$\beta$ T")
    text = text.replace("CD4-positive, alpha-beta T cell", r"CD4 T")
    # text = text.replace("CD8-positive, alpha-beta T cell", r"CD8, $\alpha$-$\beta$ T")
    text = text.replace("CD8-positive, alpha-beta T cell", r"CD8 T")
    text = text.replace("conventional dendritic cell", "cDC")
    text = text.replace("natural killer cell", "NK")
    if not text.startswith("NK"):
        text = text.split("(")[0]
    else:
        #text = text.replace("CD56-bright_NK", "CD56hi")
        #text = text.replace("CD56-dim_NK", "CD56low")
        #text = text.replace("NK_CD56hi", "CD56hi")
        text = text.replace("NK_CD56hi", "hi")
        #text = text.replace("NK_CD56low", "CD56low")
        text = text.replace("NK_CD56low", "low")
    if text.startswith("non-classical monocyte"):
        text =  "nc. Mono."
    elif text.startswith("classical monocyte"):
        text =  "c. Mono."
    return text

In [ ]:
THRESHOLD_EXPRESSION = 0.05

In [ ]:
all_results_blood = pd.read_csv("./results_final_DEGs/results_MS_blood_robustness_new.csv", index_col=0)
all_results_blood = all_results_blood.loc[all_results_blood.threshold_expression >= THRESHOLD_EXPRESSION]

In [ ]:
all_results_csf = pd.read_csv("./results_final_DEGs/results_MS_csf_robustness_new.csv", index_col=0)
all_results_csf = all_results_csf.loc[all_results_csf.threshold_expression >= THRESHOLD_EXPRESSION]

In [ ]:
all_results_blood.loc[:, "tissue"] = "blood"
all_results_csf.loc[:, "tissue"] = "csf"
all_results = pd.concat([all_results_blood, all_results_csf], axis=0, ignore_index=True)

In [ ]:
all_results.percent_duplicate_ranking_stats.max()

In [ ]:
THRESHOLD_DUPLICATE_PERCENTAGE = 0.05

num_total_results = all_results.shape[0]
num_valid_results = all_results.loc[all_results.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATE_PERCENTAGE].shape[0]
print(num_valid_results, num_total_results, num_valid_results / num_total_results)

In [ ]:
all_results = all_results.loc[all_results.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATE_PERCENTAGE]

In [ ]:
THRESHOLD_VALID = 1/3
THRESHOLD_BELOW_ALPHA = 2/3

FONTSIZE = 10

In [ ]:
results_pivot_blood, df_annot_blood, results_robust_blood = get_robust_significant_results(
    all_results_blood, 
    alphas=[0.1, 0.05, 0.01],
    threshold_valid = THRESHOLD_VALID,
    threshold_below_alpha = THRESHOLD_BELOW_ALPHA,
)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8, 8))
sns.heatmap(
    results_pivot_blood,
    square=True,
    annot=df_annot_blood,
    cmap="RdBu_r",
    center=0,
    # vmin=vmin,
    # vmax=vmax,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.5,
    cbar=False,
    ax=ax,
)

plt.xlabel("")
plt.ylabel("")
ax.set_facecolor("lightgray")
ax.tick_params(axis='both', which='both', length=0)

celltype_labels = format_celltype_names(results_pivot_blood.columns)
plt.xticks(0.5+np.arange(results_pivot_blood.shape[1]), celltype_labels, fontsize=FONTSIZE, family="Arial", rotation=90, ha="center")

cytokine_labels = format_cytokine_names(results_pivot_blood.index)
plt.yticks(0.5+np.arange(results_pivot_blood.shape[0]), cytokine_labels, fontsize=FONTSIZE, family="Arial", rotation=0, ha="right")

plt.show()

In [ ]:
def get_senders(
    adata: AnnData,
    cytokine_info: pd.DataFrame,
    cytokine: str = "IL-32-beta",
    show: bool = False,
    column_cell_type: str = "labels",
) -> pd.DataFrame:

    genes = np.unique(re.split(", ", cytokine_info.loc[cytokine_info.name == cytokine, "gene"].values[0]))
    mask = np.isin(genes, adata.var_names)

    if not mask.any():
        print(f"None of the cytokine producing genes ({genes}) were found in dataset for cytokine {cytokine}.")
        return None
    if not mask.all():
        print(f"The following cytokine producing genes were not found in the dataset and are excluded: {genes[~mask]}")
        genes = genes[mask]

    adata = adata[:, genes]
    
    adata_out = sc.tl.rank_genes_groups(
        adata,
        groupby=column_cell_type,
        copy=True,
        use_raw=False,
        method="wilcoxon",
    )
    result = adata_out.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    
    all_df = []
    for g in groups:
        df = pd.DataFrame({
            'gene': result['names'][g],
            'logfoldchanges': result['logfoldchanges'][g],
            'pvals': result['pvals'][g],
            'pvals_adj': result['pvals_adj'][g],
            column_cell_type: g,
        })
        all_df.append(df)
    all_df = pd.concat(all_df, axis=0)
    all_df.set_index(column_cell_type, inplace=True)

    obs = adata_out.obs.copy()
    assert all(obs.index == adata.obs.index)
    if not isinstance(adata.X, np.ndarray):
        obs.loc[:, "X"] = np.array(adata.X.todense()).squeeze()
    else:
        obs.loc[:, "X"] = adata.X.squeeze()
    
    obs.loc[:, "is_expressed"] = obs.X > 0
    obs.set_index(column_cell_type, inplace=True)

    a = obs.groupby("labels", observed=True).is_expressed.mean().to_frame().rename({"is_expressed": "frac_X"}, axis=1)
    b = obs.groupby("labels", observed=True).X.mean().to_frame().rename({"X": "mean_X"}, axis=1)
    c = obs.loc[obs.X > 0].groupby("labels", observed=True).X.mean().to_frame().rename({"X": "mean_X>0"}, axis=1)

    all_df = pd.concat([all_df, a, b, c], axis=1)
    all_df.loc[:, "mean_X>0"] = all_df.loc[:, "mean_X>0"].fillna(0)
    all_df.loc[:, "cytokine"] = cytokine
    return all_df


def get_receivers(
    adata: AnnData, 
    cytokine_info: pd.DataFrame,
    cytokine: str,
    column_cell_type: str = "labels"
) -> pd.DataFrame | None:

    # get receptor genes for this cytokine
    _receptor_genes = cytokine_info.loc[cytokine_info.name == cytokine, "receptor gene"]
    if _receptor_genes.isna().all():
        print(f"No receptor gene found in cytokine_info for cytokine: {cytokine}")
        return None
    assert len(_receptor_genes) == 1, _receptor_genes
    _receptor_genes = _receptor_genes.values[0]

    # there can be multiple receptors
    candidates = re.split("; ", _receptor_genes)
    
    results_mean, results_frac = [], []
    # each receptor may require the expression of multiple genes
    for candidate in candidates:
        genes = np.array(re.split(", ", candidate))
        mask = np.isin(genes, adata.var_names)
        if not mask.any():
            print(f"None of the cytokine receptor genes ({genes}) were found in dataset for cytokine {cytokine}.")
            continue
        if not mask.all():
            print(f"The following cytokine receptor genes were not found in the dataset and are excluded: {genes[~mask]}")
            genes = genes[mask]
        X_df = adata[:, genes].to_df()
        frac_df = X_df > 0
        X_df.loc[:, column_cell_type] = adata.obs.loc[:, column_cell_type].values
        frac_df.loc[:, column_cell_type] = adata.obs.loc[:, column_cell_type].values
        # take minimum average gene expression across all genes required for this receptor
        results_mean.append(X_df.groupby(column_cell_type, observed=False).mean().min(axis=1).to_frame())
        # take minimum expression fraction across all genes required for this receptor
        results_frac.append(frac_df.groupby(column_cell_type, observed=False).mean().min(axis=1).to_frame())
    if len(results_mean) == 0:
        return None

    
    results_mean = pd.concat(results_mean, axis=1).max(axis=1).to_frame().rename({0: "mean_X"}, axis=1)
    results_frac = pd.concat(results_frac, axis=1).max(axis=1).to_frame().rename({0: "frac_X"}, axis=1)
    results = pd.concat([results_mean, results_frac], axis=1)
    results.loc[:, "cytokine"] = cytokine
    return results


def get_senders_and_receivers(
    adata: AnnData, 
    cytokine_info: pd.DataFrame,
    cytokine: str,
    column_cell_type: str = "labels",
    sender_pvalue_threshold: float = 0.1,
    receiver_mean_X_threshold: float = 0,
) -> (pd.DataFrame, pd.DataFrame):

    df_senders = get_senders(adata=adata, cytokine_info=cytokine_info, cytokine=cytokine, column_cell_type=column_cell_type)
    df_receivers = get_receivers(adata=adata, cytokine_info=cytokine_info, cytokine=cytokine, column_cell_type=column_cell_type)
    if df_senders is not None:
        df_senders = df_senders.loc[(df_senders.pvals < sender_pvalue_threshold) & (df_senders.logfoldchanges > 0)]
    if df_receivers is not None:
        df_receivers = df_receivers.loc[df_receivers.mean_X > receiver_mean_X_threshold]
    return df_senders, df_receivers


def plot_communication(
    df_src: pd.DataFrame,
    df_tgt: pd.DataFrame,
    frac_expressing_cells_sender: float | None = 0.05,
    frac_expressing_cells_receiver: float | None = 0.05,
    mean_cytokine_gene_expression_sender: float | None = None,
    mean_cytokine_gene_expression_receiver: float | None = None,
    df_enrichment: pd.DataFrame | None = None,
    all_celltypes: list | None = None,
    cytokine2color: dict | None = None,
    celltype2color: dict | None = None,
    figsize: Tuple[float, float] = (5,5),
    show_legend: bool = True,
    save_path: str | None = None,
    lw: float = 1.0,
    fontsize: int = 6,
):
    """
    Generates a Circos plot to visualize cell-cell communication based on cytokine
    producer and receiver statistics.

    Filters the input dataframes based on provided thresholds for fraction of
    expressing cells and mean cytokine gene expression. It then creates a
    circular layout with cell type partitions and draws directed links
    representing cytokine communication between producer and receiver cell types.

    Parameters
    ----------
    df_src : pd.DataFrame
        DataFrame containing producer cell type and cytokine expression statistics,
        typically from `_get_expression_stats`. Must have 'celltype', 'cytokine',
        'mean_cytokine_gene_expression', and 'frac_expressing_cells' columns.
    df_tgt : pd.DataFrame
        DataFrame containing receiver cell type and cytokine expression statistics,
        typically from `_get_expression_stats`. Must have 'celltype', 'cytokine',
        'mean_cytokine_gene_expression', and 'frac_expressing_cells' columns.
    frac_expressing_cells : float | None, default 0.05
        Minimum fraction of cells expressing a cytokine/receptor gene for an
        interaction to be considered. If None, no filtering is applied based on this.
    mean_cytokine_gene_expression : float | None, default None
        Minimum mean expression of a cytokine/receptor gene for an interaction
        to be considered. If None, no filtering is applied based on this.

    """
    
    if frac_expressing_cells_sender is not None:
        df_src = df_src.loc[df_src.frac_X > frac_expressing_cells_sender]
    if frac_expressing_cells_receiver is not None:
        df_tgt = df_tgt.loc[df_tgt.frac_X > frac_expressing_cells_receiver]
    if mean_cytokine_gene_expression_sender is not None:
        df_src = df_src.loc[df_src.mean_X > mean_cytokine_gene_expression_sender]
    if frac_expressing_cells_receiver is not None:
        df_tgt = df_tgt.loc[df_tgt.mean_X > mean_cytokine_gene_expression_receiver]

    if all_celltypes is None:
        all_celltypes = sorted(np.union1d(df_src.celltype.unique(), df_tgt.celltype.unique()))
    # celltype_colors = all_palettes["Set3"][len(all_celltypes)]
    if celltype2color is None:
        celltype_colors = all_palettes["Category20"][len(all_celltypes)]
        celltype2color = dict(zip(all_celltypes, celltype_colors))
    
    all_cytokines = np.union1d(df_src.cytokine.unique(), df_tgt.cytokine.unique())
    cytokine2idx = {cytokine: k for k, cytokine in enumerate(all_cytokines)}
    # cytokine_colors = all_palettes["Category20"][len(all_cytokines)]
    # cytokine2color = dict(zip(all_cytokines, cytokine_colors))

    unique_cytokines = df_src.cytokine.unique()
    if df_enrichment is not None:
        significant_cytokines = df_enrichment.cytokine.unique()
        unique_cytokines = np.intersect1d(unique_cytokines, significant_cytokines)

    if cytokine2color is None:
        cytokine_colors = all_palettes["Colorblind"][max(3, len(unique_cytokines))]
        # cytokine_colors = all_palettes["Set3"][max(3, len(unique_cytokines))]
        cytokine2color = dict(zip(unique_cytokines, cytokine_colors))
    
    # draw outer circle / cell type partitions
    sectors = dict(zip(all_celltypes, (2*len(all_cytokines)+3)*np.ones(len(all_celltypes))))
    
    circos = Circos(sectors, space=3)
    for sector in circos.sectors:
    
        start, stop = sector.deg_lim
        center = (start + stop) / 2
        track = sector.add_track((92, 100))
    
        if 160 >= center >= 20:
            ha = "left" 
        elif 340 >= center >= 200:
            ha = "right"
        else:
            ha= "center"

        if center < 90 or center > 270:
            va = "bottom"
        else:
            va = "top"
        
        track.axis(facecolor=celltype2color[sector.name])
        # track.text(shorten_cell_type_names(sector.name), color="black", size=6, r=110, rotation="horizontal", adjust_rotation=False, family="Arial", ha=ha)
        track.text(sector.name, color="black", size=fontsize, r=110, rotation="horizontal", adjust_rotation=False, family="Arial", ha=ha, va=va)
    
    # draw links
    legend_cytokine2color = {}
    for row_idx, row in df_src.iterrows():
        src_celltype = row.celltype
        cytokine_idx = cytokine2idx[row.cytokine]
        tgt_celltypes = df_tgt.loc[df_tgt.cytokine == row.cytokine].celltype.unique()
        
        for tgt_celltype in tgt_celltypes:

            is_enriched = True # default --> plot if enriched or whenever no enrichment info is provided
            
            if df_enrichment is not None:
                select = (df_enrichment.celltype == tgt_celltype) & (df_enrichment.cytokine == row.cytokine)
                is_enriched = df_enrichment.loc[select].shape[0] > 0
            
            if is_enriched:

                linestyle = None
                _score = df_tgt.loc[(df_tgt.cytokine == row.cytokine) & (df_tgt.celltype == tgt_celltype), "mean_X"].values
                assert len(_score) == 1
                if not np.isfinite(_score[0]):
                    linestyle = "--"
                
                arrow = circos.link_line(
                    (src_celltype, 1+cytokine_idx), # src node
                    (tgt_celltype, 2+len(all_cytokines)+cytokine_idx), # tgt node
                    direction=1, 
                    color=cytokine2color[row.cytokine], 
                    # color=celltype2color[src_celltype], 
                    lw=lw,
                    arrow_height = 8.0,
                    arrow_width = 8.0,
                    linestyle = linestyle,
                )
                if not row.cytokine in legend_cytokine2color.keys():
                    legend_cytokine2color[row.cytokine] = cytokine2color[row.cytokine]
    
                
    fig = circos.plotfig(figsize=figsize)
    ax = plt.gca()
    
    legend_handles = []
    legend_labels = []
    for cytokine, color in legend_cytokine2color.items():
        legend_handles.append(mlines.Line2D([], [], color=color, lw=1.5))
        legend_labels.append(cytokine)
    if show_legend:
        lgnd = plt.legend(
            handles=legend_handles, 
            labels=legend_labels,
            title='Cytokines',
            loc='upper left',
            bbox_to_anchor=(1, 1),
            prop={'family': 'Arial', 'size': 6},
            title_fontsize=6,
        )
    plt.tight_layout()
    if save_path:
        plt.savefig(
            save_path,
            bbox_inches="tight",
            pad_inches=0,
            transparent=True,
            dpi=400,
        )
    plt.show()
    
    return legend_handles, legend_labels

### Load and prepare dataset

In [ ]:
adata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/08_Multiple_Sclerose/Schafflick20_MS_CSF.h5ad")
# subset to blood only
adata = adata[adata.obs.CSF == "False"]
invalid_labels = ["Mono Doublet", "contamination1", "B cell doublets", "doublet"]
adata = adata[~adata.obs.labels.isin(invalid_labels)]

In [ ]:
cytokine_info = "/Users/soeren.becker/repos/HumanCytokineDict/annotations/20250125_cytokine_info_with_functional_classification_LV.xlsx"
cytokine_info = pd.read_excel(cytokine_info)
results_robust_blood.loc[:, "celltype"] = results_robust_blood.celltype_combo.apply(lambda x: x.split(" (")[0])
all_celltypes = [
    'B1',
    'B2',
    'CD4',
    'CD8a',
    'CD8n',
    'Tregs',
    'Tdg',
    'Gran',
    'MegaK',
    'RBC',
    'Mono',
    'ncMono',
    'NK1',
    'NK2',
    'mDC1',
    'mDC2',
    'pDC',
    'plasma',
]

cytokine_colors = dict(zip(results_robust_blood.cytokine.unique(), all_palettes["Muted"][results_robust_blood.cytokine.nunique()]))

cell_type_colors = {
    # Dendritic Cells (Blues)
    "pDC": "#aecbe8",    # medium blue
    "mDC1": "#5b9bd5",   # darker blue
    "mDC2": "#2e75b6",   # light blue

    # B Cells (Greens)
    "B1": "#4caf50",     # medium green
    "B2": "#81c784",     # light green
    "plasma": "#2e7d32", # dark green

    # T Cells (Oranges)
    "CD4": "#fb8c00",    # deep orange
    "CD8a": "#ef6c00",   # light orange
    "CD8n": "#e65100",   # darker orange
    "Tdg": "#ffa726",    # medium orange
    "Tregs": "#ffcc80",  # pale orange

    # NK Cells (Reds)
    "NK1": "#e53935",    # medium red
    "NK2": "#ef5350",    # light red

    # Monocytes (Purples/Browns)
    "Mono": "#8d6e63",   # brownish
    "ncMono": "#a1887f", # lighter brown

    # RBCs and Granulocytes (Grays/Pinks)
    "RBC": "#d81b60",    # pink-red
    "Gran": "#f06292",   # light pink

    # Megakaryocytes (Purples)
    "MegaK": "#ba68c8",  # lavender purple
}

### determine potential cytokine senders and receivers

In [ ]:
senders, receivers = [], []

for cytokine in results_robust_blood.cytokine.unique():
    df_senders, df_receivers = get_senders_and_receivers(
        adata = adata, 
        cytokine_info = cytokine_info,
        cytokine = cytokine,
        column_cell_type = "labels",
        sender_pvalue_threshold = 0.1,
        receiver_mean_X_threshold = 0,
    )

    if cytokine == "IL-32-beta":
        # no known receptor genes
        df_receivers = pd.DataFrame.from_dict(
            dict(zip(all_celltypes, np.ones([len(all_celltypes), 2])*np.inf)), 
            orient="index",
        ).rename({0: "mean_X", 1: "frac_X"}, axis=1)
        df_receivers.loc[:, "cytokine"] = cytokine
    
    if df_senders is not None and df_receivers is not None:
        df_senders.loc[:, "celltype"] = df_senders.index
        df_receivers.loc[:, "celltype"] = df_receivers.index
    
        senders.append(df_senders)
        receivers.append(df_receivers)

df_src = pd.concat(senders)
df_tgt = pd.concat(receivers)

In [ ]:
non_log1p_transformed_counts = np.asarray(np.sum(np.exp(adata.X.todense())-1, axis=1)).squeeze()
print(non_log1p_transformed_counts.min(), non_log1p_transformed_counts.max()) # data is normalized to 10k counts per cell

In [ ]:
fig_path = "./results_final_DEGs/figures"

expression_threshold = 0.1/10_000 # = 100 TPM
expression_threshold = np.log2((expression_threshold) + 1)

legend_handles, legend_labels = plot_communication(
    df_src = df_src,
    df_tgt = df_tgt,
    frac_expressing_cells_sender = 0,
    frac_expressing_cells_receiver = 0,
    mean_cytokine_gene_expression_sender = expression_threshold,
    mean_cytokine_gene_expression_receiver = expression_threshold,
    df_enrichment=results_robust_blood,
    all_celltypes=np.array(all_celltypes),
    cytokine2color=cytokine_colors,
    celltype2color=cell_type_colors,
    figsize=(1.9, 1.9),
    show_legend=False,
    save_path = os.path.join(fig_path, "communication_blood.svg"),
    lw=0.5,
    fontsize=6,
)

In [ ]:
fig_path = "./results_final_DEGs/figures"

expression_threshold = 1/10_000 # = 100 TPM
expression_threshold = np.log2((expression_threshold) + 1)

legend_handles, legend_labels = plot_communication(
    df_src = df_src,
    df_tgt = df_tgt,
    frac_expressing_cells_sender = 0,
    frac_expressing_cells_receiver = 0,
    mean_cytokine_gene_expression_sender = expression_threshold,
    mean_cytokine_gene_expression_receiver = expression_threshold,
    df_enrichment=results_robust_blood,
    all_celltypes=np.array(all_celltypes),
    cytokine2color=cytokine_colors,
    celltype2color=cell_type_colors,
    figsize=(1.9, 1.9),
    show_legend=False,
    save_path = os.path.join(fig_path, "communication_blood.svg"),
    lw=0.5,
    fontsize=6,
)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(0.3, 0.3))

lgnd = plt.legend(
    handles=legend_handles, 
    labels=[replace_greek(l) for l in legend_labels],
    #title='Cytokines',
    prop={'family': 'Arial', 'size': 6},
    title_fontsize=6,
    ncol=2,
)
ax.set_axis_off()
plt.tight_layout()
if True:
    plt.savefig(
        os.path.join(fig_path, "communication_legend.svg"),
        bbox_inches="tight",
        pad_inches=0,
        transparent=True,
        dpi=400,
    )
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(0.3, 0.3))

lgnd = plt.legend(
    handles=legend_handles, 
    labels=[replace_greek(l) for l in legend_labels],
    #title='Cytokines',
    prop={'family': 'Arial', 'size': 6},
    title_fontsize=6,
    ncol=4,
)
ax.set_axis_off()
plt.tight_layout()
if True:
    plt.savefig(
        os.path.join(fig_path, "communication_legend.svg"),
        bbox_inches="tight",
        pad_inches=0,
        transparent=True,
        dpi=400,
    )
plt.show()